# Map-Reduce Summarization and Confidence

Map-reduce summarizes a large collection in two stages: extract evidence from each document, then combine related evidence into broader conclusions.


## What to learn

- Map: process one interview and return a theme, source id, and exact quote.
- Verify: confirm that displayed quotes exist in the source.
- Reduce: group observations by theme and summarize each group.
- Confidence: show respondent counts and evidence strength instead of a model-generated percentage.

## Decision rule

Use one interview as the map unit when it fits. Count distinct respondents, keep prevalence separate from severity, and label conclusions based on very few responses as emerging signals.


## Runnable example

Run the following cells, then change one input and observe what changes. Focus on the decision being demonstrated rather than framework-specific syntax.


In [ ]:
"""Map-reduce synthesis with support counts, quote verification, and tiers."""
# Import the dependencies used by this example.
import json

TRANSCRIPTS = {
    "i1": "Honestly the signup flow kept erroring out. I nearly gave up.",
    "i2": "Creating my account took three tries. Once in, it was fine.",
    "i3": "Pricing tiers are confusing, I do not know what I am paying for.",
    "i4": "The signup page crashed twice on my phone.",
}

# --- MAP: per-interview structured extraction (cheap model in prod) -------
MAP_OUTPUTS = [
    {"id": "i1", "theme": "onboarding friction", "quote": "the signup flow kept erroring out"},
    {"id": "i2", "theme": "onboarding friction", "quote": "took three tries"},
    {"id": "i3", "theme": "pricing confusion", "quote": "Pricing tiers are confusing"},
    {"id": "i4", "theme": "onboarding friction", "quote": "signup page crashed twice"},
    # A hallucinated quote, to show verification catching it:
    {"id": "i3", "theme": "pricing confusion", "quote": "I would pay double for clarity"},
]


# Define the data shape and small operations before running them.
def verify_quote(interview_id: str, quote: str) -> bool:
    """Cheap anti-hallucination check: the quote must exist in the source."""
    return quote.lower() in TRANSCRIPTS.get(interview_id, "").lower()


verified = [m for m in MAP_OUTPUTS if verify_quote(m["id"], m["quote"])]
rejected = [m for m in MAP_OUTPUTS if m not in verified]
print(f"verified {len(verified)}/{len(MAP_OUTPUTS)} quotes; rejected: "
      f"{[m['quote'] for m in rejected]}")

# --- REDUCE: per-theme synthesis, citation-constrained to map outputs -----
N_TOTAL = len(TRANSCRIPTS)
themes = {}
for m in verified:
    themes.setdefault(m["theme"], []).append(m)


def tier(n: int, total: int) -> str:
    share = n / total
    if n >= 3 and share >= 0.5:
        return "established"
    if n >= 2:
        return "supported"
    return "emerging signal (low n)"


insights = [
    {
        "theme": theme,
        "support": f"{len(set(m['id'] for m in members))} of {N_TOTAL} respondents",
        "tier": tier(len(set(m["id"] for m in members)), N_TOTAL),
        "evidence": [{"id": m["id"], "quote": m["quote"]} for m in members],
    }
    for theme, members in themes.items()
]
print(json.dumps(insights, indent=2))


## Study questions

1. What problem does this pattern solve?
2. What is the simplest alternative?
3. Which failure would tell you that the pattern is implemented incorrectly?
4. What measurement would justify adding more complexity?

## Before using it

- [ ] The requirement and success measure are explicit.
- [ ] Inputs and outputs are validated.
- [ ] Failures, retries, and stopping conditions are bounded.
- [ ] Security and privacy match the consequences of the action.
- [ ] The behavior is tested on realistic examples.
